# Ejercicio 2 – MEC (2048)

Este notebook sirve como guía y bitácora para documentar el **Ejercicio 2** del obligatorio de Inteligencia Artificial.

Objetivo (según `LetraObligatorio.md`): implementar **Expectimax** y **Minimax con poda α‑β**, diseñar una **función de evaluación** con varias heurísticas, y **experimentar** (combinaciones + ponderaciones) dejando un registro claro de resultados.


## 1. Entendimiento del problema

**2048** es un juego de tablero 4×4 donde en cada turno:
1) El jugador elige un movimiento (`UP=0`, `DOWN=1`, `LEFT=2`, `RIGHT=3`).
2) Se inserta una ficha aleatoria: `2` con prob. 0,9 o `4` con prob. 0,1 en una celda vacía.

El agente busca maximizar la performance (idealmente llegar a 2048), manteniendo el tablero jugable.


## 2. Técnicas implementadas

### 2.1 Expectimax
- Turno del jugador: nodo **MAX**.
- Inserción aleatoria: nodo **CHANCE** (valor esperado sobre todas las celdas vacías y fichas 2/4).
- Corte por profundidad (`depth`) y evaluación heurística en hojas.

### 2.2 Minimax con poda α‑β
- Turno del jugador: nodo **MAX**.
- Inserción: nodo **MIN** adversarial (elige la peor inserción) para aproximar “peor caso”.
- Poda **α‑β** opcional (y se mide el impacto con/sin poda).

### Nota importante (reproducibilidad)
En este repo se agregó un wrapper (`preserve_numpy_rng`) para que llamadas internas del template (`clone()` y `get_available_moves()`) no consuman aleatoriedad y no alteren la secuencia real del juego durante la búsqueda.


## 3. Función de evaluación (heurísticas)

La evaluación combina heurísticas ponderadas (ver `heuristics.py`). Ejemplos:
- `count_empty`
- `monotonicity`
- `smoothness`
- `max_tile_in_corner`
- `merges_possible`
- `positional_weight` (serpiente)

Para experimentar con “distintas combinaciones y ponderadas de distintas formas”, usamos **presets** definidos en `bench.py` (por ejemplo: `baseline`, `smooth_heavy`, `snake`).


## 4. Matriz final de experimentos (lo que vamos a correr)

Esta matriz cubre lo que pide la letra:
- Comparación de **técnicas**: Expectimax vs Minimax (con poda).
- Comparación de **ponderaciones**: presets representativos (3).
- Impacto de **poda α‑β**: Minimax con y sin poda.
- Baseline: Random.

Usamos `--seed 0` para reproducibilidad.


### 4.1 Preparación

Ejecutar desde la carpeta `2048/`.

Crear carpeta de resultados (elegí el que corresponda a tu terminal):

- Bash / WSL:
```bash
mkdir -p results
```
- PowerShell:
```bash
New-Item -ItemType Directory -Force results | Out-Null
```
- Alternativa (cross-platform):
```bash
python -c "from pathlib import Path; Path('results').mkdir(exist_ok=True)"
```


### 4.2 Comandos (sin continuaciones de línea)

#### Ya ejecutado (según tu bitácora)

```bash
poetry run python bench.py --agent random --episodes 10 --seed 0 --output results/random_e10_seed0.csv
poetry run python bench.py --agent expectimax --depth 3 --preset baseline --episodes 10 --seed 0 --output results/expectimax_baseline_d3_e10_seed0.csv
```

#### Pendiente (matriz base)

Expectimax (ponderaciones / presets, `depth=3`):
```bash
poetry run python bench.py --agent expectimax --depth 3 --preset smooth_heavy --episodes 10 --seed 0 --output results/expectimax_smooth_heavy_d3_e10_seed0.csv
poetry run python bench.py --agent expectimax --depth 3 --preset snake --episodes 10 --seed 0 --output results/expectimax_snake_d3_e10_seed0.csv
```

Expectimax (tradeoff profundidad/tiempo, mismo preset):
```bash
poetry run python bench.py --agent expectimax --depth 2 --preset smooth_heavy --episodes 10 --seed 0 --output results/expectimax_smooth_heavy_d2_e10_seed0.csv
```

Minimax (con poda, para comparar técnica vs Expectimax):
```bash
poetry run python bench.py --agent minimax --depth 3 --preset smooth_heavy --episodes 10 --seed 0 --output results/minimax_smooth_heavy_d3_prune_on_e10_seed0.csv
```

Impacto poda α‑β (comparación justa 5 vs 5):
```bash
poetry run python bench.py --agent minimax --depth 3 --preset smooth_heavy --episodes 5 --seed 0 --output results/minimax_smooth_heavy_d3_prune_on_e5_seed0.csv
poetry run python bench.py --agent minimax --depth 3 --preset smooth_heavy --episodes 5 --seed 0 --no-pruning --output results/minimax_smooth_heavy_d3_prune_off_e5_seed0.csv
```

Minimax (sensibilidad a ponderaciones, mínimo extra):
```bash
poetry run python bench.py --agent minimax --depth 3 --preset baseline --episodes 5 --seed 0 --output results/minimax_baseline_d3_prune_on_e5_seed0.csv
```

#### Matriz extendida (opcional, mejora la justificación)

Minimax (más presets, siempre con poda ON para que sea corrible):
```bash
poetry run python bench.py --agent minimax --depth 3 --preset snake --episodes 5 --seed 0 --output results/minimax_snake_d3_prune_on_e5_seed0.csv
poetry run python bench.py --agent minimax --depth 3 --preset corner_heavy --episodes 5 --seed 0 --output results/minimax_corner_heavy_d3_prune_on_e5_seed0.csv
```

Profundidad (Minimax, con poda ON):
```bash
poetry run python bench.py --agent minimax --depth 2 --preset smooth_heavy --episodes 5 --seed 0 --output results/minimax_smooth_heavy_d2_prune_on_e5_seed0.csv
```

Profundidad (Expectimax, opcional si da el tiempo):
```bash
poetry run python bench.py --agent expectimax --depth 4 --preset smooth_heavy --episodes 5 --seed 0 --output results/expectimax_smooth_heavy_d4_e5_seed0.csv
```

Robustez (repetir con más semillas la mejor config de cada técnica):
```bash
poetry run python bench.py --agent expectimax --depth 3 --preset smooth_heavy --episodes 5 --seed 1 --output results/expectimax_smooth_heavy_d3_e5_seed1.csv
poetry run python bench.py --agent expectimax --depth 3 --preset smooth_heavy --episodes 5 --seed 2 --output results/expectimax_smooth_heavy_d3_e5_seed2.csv
poetry run python bench.py --agent minimax --depth 3 --preset smooth_heavy --episodes 5 --seed 1 --output results/minimax_smooth_heavy_d3_prune_on_e5_seed1.csv
poetry run python bench.py --agent minimax --depth 3 --preset smooth_heavy --episodes 5 --seed 2 --output results/minimax_smooth_heavy_d3_prune_on_e5_seed2.csv
```


## 5. Resumen de resultados (tabla)

Opción A (consola):
```bash
poetry run python summarize_results.py --pattern "results/*.csv"
```

Opción B (en notebook): correr la celda siguiente para agrupar y comparar.


In [ ]:
import pandas as pd
from pathlib import Path
import re


def load_results(path_pattern: str = "results/*.csv") -> pd.DataFrame:
    files = sorted(Path('.').glob(path_pattern))
    dfs = []
    for f in files:
        df = pd.read_csv(f)
        df["file"] = f.name
        dfs.append(df)
    return pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()


results_df = load_results()
if results_df.empty:
    print("No se encontraron resultados en results/*.csv")
else:
    presets = ["baseline", "tuned", "corner_heavy", "smooth_heavy", "snake"]

    def parse_info(filename: str):
        name = filename
        agent = (
            "expectimax" if "expectimax" in name else
            "minimax" if "minimax" in name else
            "random" if "random" in name else None
        )

        depth = None
        m_depth = re.search(r"_d(\d+)_", name)
        if m_depth:
            depth = int(m_depth.group(1))

        preset = None
        for p in presets:
            if p in name:
                preset = p
                break

        variant = None
        if "prune_on" in name:
            variant = "prune_on"
        elif "prune_off" in name:
            variant = "prune_off"

        seed = None
        m_seed = re.search(r"_seed(\d+)\.csv$", name)
        if m_seed:
            seed = int(m_seed.group(1))

        return agent, depth, preset, variant, seed

    parsed = results_df["file"].apply(parse_info)
    results_df[["agent", "depth", "preset", "variant", "seed"]] = pd.DataFrame(parsed.tolist(), index=results_df.index)

    results_df["win"] = results_df["win"].astype(str).str.lower().isin(["true", "1", "yes"])
    results_df["max_tile"] = pd.to_numeric(results_df["max_tile"], errors="coerce")
    results_df["moves"] = pd.to_numeric(results_df["moves"], errors="coerce")
    results_df["duration_sec"] = pd.to_numeric(results_df["duration_sec"], errors="coerce")

    summary = (
        results_df.groupby(["agent", "depth", "preset", "variant", "seed"], dropna=False)
        .agg(
            episodes=("win", "count"),
            win_rate=("win", "mean"),
            avg_max_tile=("max_tile", "mean"),
            avg_moves=("moves", "mean"),
            avg_duration_sec=("duration_sec", "mean"),
        )
        .reset_index()
        .sort_values(["agent", "preset", "depth", "variant", "seed"], na_position="last")
    )

    display(summary)

    # Agregado extra: promedio sobre seeds (útil para justificar robustez)
    summary_over_seeds = (
        summary.groupby(["agent", "depth", "preset", "variant"], dropna=False)
        .agg(
            seeds=("seed", "nunique"),
            episodes_total=("episodes", "sum"),
            win_rate_avg=("win_rate", "mean"),
            avg_max_tile_avg=("avg_max_tile", "mean"),
            avg_moves_avg=("avg_moves", "mean"),
            avg_duration_sec_avg=("avg_duration_sec", "mean"),
        )
        .reset_index()
        .sort_values(["agent", "preset", "depth", "variant"], na_position="last")
    )

    display(summary_over_seeds)


## 6. Conclusiones (qué responder en el informe)

Checklist de puntos a cubrir:
- ¿Qué técnica rindió mejor (Expectimax vs Minimax)? ¿En qué métrica (win_rate, max_tile, tiempo)?
- ¿Cómo impactan los presets (ponderaciones) en Expectimax?
- ¿Cuál es el impacto de la poda α‑β en Minimax? (principalmente tiempo, y si cambia calidad)
- Comparación contra baseline Random.

Nota: si alguna corrida (por ejemplo Minimax sin poda) es demasiado lenta, dejarlo documentado como resultado del experimento (costo computacional) y justificar por qué se redujo episodios.
